# Colab 3 — Use a Wrapper When Schemas Differ

**Goal:** Translate parent state into the shape expected by the subgraph, then translate the result back.


In [1]:
!pip -q install -U langgraph


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 4.5 MB/s eta 0:00:00


In [2]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

#SUBGRAPH
class ProductState(TypedDict):
    query: str
    recommendation: str

def recommend(state: ProductState):
    return {"recommendation": f"Top match for '{state['query']}': Sony WH-1000XM5"}

sub_builder = StateGraph(ProductState)
sub_builder.add_node("recommend", recommend)
sub_builder.add_edge(START, "recommend")
sub_builder.add_edge("recommend", END)
product_subgraph = sub_builder.compile()

#PARENT GRAPH
class ParentState(TypedDict):
    customer_request: str
    final_answer: str

# wrapper
def call_product_subgraph(state: ParentState):
    # Parent -> subgraph
    result = product_subgraph.invoke({
        "query": state["customer_request"],
        "recommendation": ""
    })
    # Subgraph -> parent
    return {"final_answer": result["recommendation"]}

parent_builder = StateGraph(ParentState)
parent_builder.add_node("product_agent", call_product_subgraph)
parent_builder.add_edge(START, "product_agent")
parent_builder.add_edge("product_agent", END)
graph = parent_builder.compile()

graph.invoke({
    "customer_request": "Recommend premium noise-canceling headphones",
    "final_answer": ""
})


{'customer_request': 'Recommend premium noise-canceling headphones',
 'final_answer': "Top match for 'Recommend premium noise-canceling headphones': Sony WH-1000XM5"}

### Takeway

> “The wrapper is an adapter: `customer_request → query`, then `recommendation → final_answer`.”
